In [1]:
import os
import cv2
import urllib.request
import zipfile
import numpy as np
from skimage import io
from sklearn.svm import SVC
from skimage.feature import hog
from sklearn.metrics import precision_score, accuracy_score

In [2]:
url = "https://github.com/itsmathematicsboy/Apple-and-Tomato-Classification-Image/raw/main/apple%20tomato%20data.zip"

urllib.request.urlretrieve(url, "data.zip")

with zipfile.ZipFile("data.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

In [3]:
print("Isi folder dataset:")
print(os.listdir("dataset"))

Isi folder dataset:
['test', 'train']


In [4]:
train_path = "dataset/train"
test_path  = "dataset/test"

In [5]:
print('Train Folder:')
print(os.listdir(train_path))
print('Test Folder')
print(os.listdir(test_path))

Train Folder:
['apples', 'tomatoes']
Test Folder
['apples', 'tomatoes']


In [6]:
# Train Data
def get_train_data(train_path):
    X = []
    y = []
    
    for label, folder in enumerate(sorted(os.listdir(train_path))):
        folder_path = os.path.join(train_path, folder)
        
        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                img_path = os.path.join(folder_path, file)

                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    X.append(img_path)
                    y.append(label)
    
    return np.array(X), np.array(y)

# Test Data
def get_test_data(test_path):
    X = []
    y = []
    
    for label, folder in enumerate(sorted(os.listdir(test_path))):
        folder_path = os.path.join(test_path, folder)
        
        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                img_path = os.path.join(folder_path, file)
                
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    X.append(img_path)
                    y.append(label)
    
    return np.array(X), np.array(y)

In [7]:
# Getting train and testing data
x_train, y_train = get_train_data(train_path)
x_test, y_test = get_test_data(test_path)

In [8]:
# Count the unique data
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'Total Data For {u} Labels = {c}')

Total Data For 0 Labels = 164
Total Data For 1 Labels = 130


In [9]:
def weight_compute(data, len_class, count_class):
    return (len(data)) / (len_class * count_class)

In [10]:
apple_class_weight = weight_compute(x_train, 2, 164)
tomato_class_weight = weight_compute(x_train, 2, 130)

In [11]:
weight_class = {0 : apple_class_weight, 1 : tomato_class_weight}

In [12]:
def image_preprocessing(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64))

    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    
    return features

In [13]:
x_train_scaled = np.array([image_preprocessing(p) for p in x_train])
x_test_scaled = np.array([image_preprocessing(p) for p in x_test])

In [14]:
model = SVC(C = 10, gamma = 0.01,class_weight = weight_class, kernel = 'rbf')

In [15]:
model.fit(x_train_scaled, y_train)

SVC(C=10, class_weight={0: 0.8963414634146342, 1: 1.1307692307692307},
    gamma=0.01)

In [16]:
y_predict = model.predict(x_test_scaled)

In [17]:
def accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def precision(y_true, y_pred):
    return precision_score(y_true, y_pred)

In [18]:
print(f'Accuracy: {accuracy(y_test, y_predict)}')
print(f'Precision: {precision(y_test, y_predict)}')

Accuracy: 0.7422680412371134
Precision: 0.7368421052631579
